## CRM Customers_info Silver Ttransformation

### 00. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from datetime import datetime

### 01. Build today's folder name and read from bronze table

In [0]:
today = datetime.today().strftime("%Y_%m_%d")   # e.g. 2026_06_23
bronze_table = f"`abc_business-core`.bronze_{today}.customers_info"
df = spark.table(bronze_table)

###  02. Trim all string fields

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

###  03. Normalisation

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

### 04. Remove missing customer IDs

In [0]:
df = df.filter(F.col("cst_id").isNotNull())

### 05. Rename columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

###  6. Preview - Sanity check

In [0]:
df.limit(5).display()

###  07. Write Silver table

In [0]:
table_name = f"`abc_business-core`.silver.crm_customers_{today}"
df.write.mode("overwrite").format("delta").saveAsTable(table_name)

### 08. Sanity checks of silver table

In [0]:
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 5")
df.show()